## Célula 2: Autenticação e Configuração

In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import orcid
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# Substitua com as suas credenciais do Developer Tools do ORCID
CLIENT_ID = os.getenv('ORCID_CLIENT_ID')
CLIENT_SECRET = os.getenv('ORCID_CLIENT_SECRET')

# Substitua pelo ORCID iD que deseja analisar (exemplo genérico)
ORCID_ID = '0000-0002-4258-0424'

# Inicializa a API Pública (sandbox=False significa que estamos acessando dados reais de produção)
api = orcid.PublicAPI(CLIENT_ID, CLIENT_SECRET, sandbox=False)

# Obtém um token de acesso de leitura pública (necessário para ler o perfil)
token = api.get_search_token_from_orcid()
print("Autenticação realizada com sucesso!")

Autenticação realizada com sucesso!


## Célula 2: Extração dos Dados do Perfil

In [4]:
# Lê o registro público completo do perfil
perfil = api.read_record_public(ORCID_ID, 'record', token)

# 1. Extração de Informações Pessoais
pessoa = perfil.get('person', {})
nome_dict = pessoa.get('name') or {}

# Utilizando .get() de forma encadeada pois muitos perfis no ORCID têm campos vazios
primeiro_nome = nome_dict.get('given-names', {}).get('value', '') if nome_dict.get('given-names') else ''
sobrenome = nome_dict.get('family-name', {}).get('value', '') if nome_dict.get('family-name') else ''
nome_completo = f"{primeiro_nome} {sobrenome}".strip()

# Biografia
bio_dict = pessoa.get('biography') or {}
biografia = bio_dict.get('content', 'Biografia não informada') if bio_dict else 'Biografia não informada'

# 2. Extração das Publicações (Works)
# Além do perfil básico, é útil extrair a produção técnica/acadêmica
atividades = perfil.get('activities-summary', {})
trabalhos_info = atividades.get('works', {}).get('group', [])

lista_trabalhos = []

for grupo in trabalhos_info:
    # Pegamos o resumo principal do trabalho
    trabalho = grupo.get('work-summary', [{}])[0]
    
    # Título e Tipo
    titulo = trabalho.get('title', {}).get('title', {}).get('value', 'Sem título')
    tipo = trabalho.get('type', 'Não especificado')
    
    # Onde foi publicado (Revista / Conferência)
    revista_dict = trabalho.get('journal-title') or {}
    revista = revista_dict.get('value', 'Não informado')
    
    # Ano de Publicação
    pub_date = trabalho.get('publication-date') or {}
    ano = pub_date.get('year', {}).get('value', 'N/D') if pub_date else 'N/D'
    
    # Identificadores Externos (Buscando o DOI especificamente)
    doi = 'Não encontrado'
    ext_ids_container = trabalho.get('external-ids', {})
    if ext_ids_container:
        ext_ids = ext_ids_container.get('external-id', [])
        for ident in ext_ids:
            if ident.get('external-id-type') == 'doi':
                doi = ident.get('external-id-value')
                break # Para no primeiro DOI encontrado
                
    # Link (URL)
    url_dict = trabalho.get('url') or {}
    url = url_dict.get('value', 'Sem URL')

    # Adicionando à lista para o DataFrame
    lista_trabalhos.append({
        'ORCID iD': ORCID_ID,
        'Título': titulo,
        'Veículo (Revista/Conferência)': revista,
        'Tipo': tipo,
        'Ano': ano,
        'DOI': doi,
        'URL': url
    })

# Quando você jogar "lista_trabalhos" no pd.DataFrame(), 
# terá colunas ricas com DOI, tipo de evento, onde foi publicado, etc.

## Célula 3: Armazenamento em DataFrames e Visualização

In [5]:
# DataFrame 1: Informações consolidadas do Perfil (Útil para sumarizar dados pessoais)
df_perfil = pd.DataFrame([{
    'ORCID iD': ORCID_ID,
    'Nome Completo': nome_completo,
    'Biografia': biografia,
    'Total de Publicações': len(lista_trabalhos)
}])

print("=== Resumo do Perfil ===")
display(df_perfil)

# DataFrame 2: Lista de Trabalhos / Produção (Útil para análises cienciométricas ou de literatura)
df_trabalhos = pd.DataFrame(lista_trabalhos)

print("\n=== Produção Acadêmica ===")
if not df_trabalhos.empty:
    display(df_trabalhos)
else:
    print("Nenhum registro de produção encontrado.")

=== Resumo do Perfil ===


,ORCID iD,Nome Completo,Biografia,Total de Publicações
0,0000-0002-4258-0424,Guilherme Horta Travassos,Full professor of Software Engineering at Copp...,69



=== Produção Acadêmica ===


,ORCID iD,Título,Veículo (Revista/Conferência),Tipo,Ano,DOI,URL
0,0000-0002-4258-0424,"Design, execution, and contextual factors shap...",Não informado,JOURNAL_ARTICLE,2026,10.1016/j.infsof.2026.108136,Sem URL
1,0000-0002-4258-0424,A Critical Reflection on the State of Data Ana...,Não informado,JOURNAL_ARTICLE,2026,10.1145/3799715,Sem URL
2,0000-0002-4258-0424,Exploring Technology Probe Applications in Sof...,Não informado,OTHER,2026,10.2139/ssrn.6443058,Sem URL
3,0000-0002-4258-0424,Experimental Evaluation of a Checklist-Based I...,Não informado,JOURNAL_ARTICLE,2025,10.1007/s10664-025-10681-7,Sem URL
4,0000-0002-4258-0424,Testing Context-Aware Software Systems From th...,Não informado,JOURNAL_ARTICLE,2025,10.1109/TII.2025.3529918,Sem URL
...,...,...,...,...,...,...,...
64,0000-0002-4258-0424,Detecting defects in Object Oriented designs: ...,Não informado,JOURNAL_ARTICLE,1999,10.1145/320385.320389,Sem URL
65,0000-0002-4258-0424,Towards an ontology of software maintenance,Não informado,JOURNAL_ARTICLE,1999,10.1002/(SICI)1096-908X(199911/12)11:6&lt;365:...,Sem URL
66,0000-0002-4258-0424,An OO Software engineering training experience...,Não informado,JOURNAL_ARTICLE,1998,Não encontrado,Sem URL
67,0000-0002-4258-0424,An approach to perform behavior testing in obj...,Não informado,JOURNAL_ARTICLE,1998,Não encontrado,Sem URL
